# 1. Import Libraries & Configuration

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from tqdm import tqdm
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

tqdm.pandas()

BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models' / 'distilbert_prod'

import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("TMDB_API_KEY")
TOTAL_PAGES = 50

c:\Users\VICTUS\OneDrive\Documents\Edwin's_Project\Moofy\emovie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Extract Movie Data from TMDB

In [2]:
movies = []

for page in tqdm(range(1, TOTAL_PAGES + 1), desc="Fetching TMDB"):
    url = (
        f"https://api.themoviedb.org/3/movie/popular"
        f"?api_key={API_KEY}&language=en-US&page={page}"
    )

    response = requests.get(url)

    if response.status_code == 200:
        for item in response.json().get('results', []):
            if item.get('overview'):
                movies.append({
                    'movie_id': item['id'],
                    'title': item['title'],
                    'genres': item.get('genre_ids', []),
                    'overview': item['overview']
                })

df_movies = (
    pd.DataFrame(movies)
    .drop_duplicates(subset='movie_id')
    .reset_index(drop=True)
)

df_movies.shape

Fetching TMDB: 100%|██████████| 50/50 [00:31<00:00,  1.57it/s]


(988, 4)

# 3. Load the DistilBERT Model

In [3]:
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_DIR)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)

with open(MODEL_DIR / 'label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2762.91it/s]


# 4. Predict Movie Emotions

In [4]:
def predict_movie_emotion(text):
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    pred_id = np.argmax(outputs.logits.numpy(), axis=1)[0]
    return label_encoder.inverse_transform([pred_id])[0]


df_movies['emotion_label'] = (
    df_movies['overview']
    .progress_apply(predict_movie_emotion)
)

100%|██████████| 988/988 [00:39<00:00, 24.81it/s]


# 5. Save the Labeled Dataset

In [5]:
OUTPUT_PATH = DATA_DIR / 'tmdb_movies_with_emotions.csv'

df_movies.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding='utf-8'
)

# 6. Preview Results

In [6]:
display(
    df_movies[
        ['title', 'overview', 'emotion_label']
    ].head(10)
)

,title,overview,emotion_label
0,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,Surprise
1,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",Fear
2,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,Fear
3,The Last House,A family suddenly sealed inside their home mus...,Fear
4,Minions & Monsters,"This is the rambunctious, ridiculous and total...",Anger
5,Colony,Professor Se-jeong is thrust into a bloody nig...,Fear
6,Obsession,"After breaking the mysterious ""One Wish Willow...",Fear
7,Toy Story 5,When Bonnie receives a Lilypad tablet as a gif...,Joy
8,The Death of Robin Hood,Grappling with his past after a life of crime ...,Sadness
9,Rage of Stars,A story about a woman from the International S...,Sadness
